## Validate Profeel EPC Estimation

**Purpose**
Checks EPC-related transformations on Profeel data and prepares standardized stock tables for matching with SDES.

**Primary inputs**
- `data/source/profeel/building_stock_profeel_detailed.csv`

**Primary outputs**
- `data/derived/profeel/building_stock_profeel_std.csv`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



# Step 3: Profeel EPC Validation

**Purpose:** Validate energy performance certificate calculations against the Profeel detailed database.

**Project imports:** `project.thermal`, `project.utils`, `project.input.param`.

**Inputs:** Profeel physical characteristics database.

**Outputs:** Validated EPC ratings for Profeel buildings.


In [ ]:
import os
from project.thermal import conventional_heating_final, stat_heating_consumption, HDD, conventional_energy_3uses ,VENTILATION_TYPES
from project.utils import make_plot
from project.input.param import generic_input
import pandas as pd
import math


# Estimation of EPC and energy need
The Notebook is looking for the origins of the error of the primary energy estimation functions used in Res-IRF when applied to the Profeel database. The output of the Notebook is the profeel database enriched by EPC estimated with Res-IRF functions used for merging household data in data_matching's Notebook.
1. Introduction
    * First estimations with thermal functions
    * Comparisons of the ratio_surface with the aggregated from Res-IRF
2. Validation: primary energy estimations
    * Estimation with the ratio_surface of Res-IRF
    * Standard estimation: what will do Res-IRF when running this database (actual ratio_surface and no ventilation)
    * Estimation with the actual ratio_surface but adding ventilation thermal transmittance values
    * Estimation weighted by the energy mix
    * Investigating some comparisons


## Introduction


In [ ]:
buildings = pd.read_csv(os.path.join('data', 'source', 'profeel', 'building_stock_profeel_detailed.csv'), index_col=['Class', 'Housing type'])
buildings = buildings[buildings['Heating system'] != 'RCU']
ratio_surface = buildings.loc[:,'Ratio_surface_Wall':'Ratio_surface_Window']
ratio_surface = ratio_surface.rename(columns={'Ratio_surface_Floor': "Floor", 'Ratio_surface_Wall': "Wall", 'Ratio_surface_Roof': "Roof", 'Ratio_surface_Window': "Windows"})


In [ ]:
'''efficiency = pd.read_csv('project/input/efficiency.csv')
efficiency = to_dict(efficiency)'''


In [ ]:
# For the moment I'm not calling the csv
efficiency = {'Electricity-Performance boiler': 0.95,
           'Natural gas-Standard boiler': 0.6,
          'Natural gas-Performance boiler': 0.76,
              'Wood fuel-Standard boiler': 0.6
           }


heating_eff = buildings['Heating system'].replace(efficiency).set_axis(buildings.index)
buildings = pd.concat((buildings, heating_eff.rename('Efficiency')), axis=1)


In [ ]:
# Could change the csv directly
translate = {'Natural ventilation': 'Ventilation naturelle', 'CMV': 'VMC SF auto et VMC double flux',
                     'CMV hygro': 'VMC SF hydrogérable'}

buildings['Ventilation'] = buildings['Ventilation'].replace(translate)
buildings['Ventilation'] = buildings['Ventilation'].replace(VENTILATION_TYPES)



In [ ]:
conventional_heating_final(buildings['Wall'], buildings['Floor'], buildings['Roof'], buildings['Windows'], ratio_surface, buildings['Efficiency'])


In [ ]:
stat_heating_consumption(buildings['Wall'], buildings['Floor'], buildings['Roof'], buildings['Windows'], buildings['Efficiency'], ratio_surface, HDD)


##### Comparing the surface ratio of Profeel's database with the aggregated ones in Res-IRF


In [ ]:
temp = pd.concat((buildings['Stock buildings'], ratio_surface), axis=1)
av_ratio_surface = (temp['Stock buildings'] * temp.loc[:, 'Wall':].T).T
av_ratio_surface = ((av_ratio_surface.groupby('Housing type').sum()).T/temp.groupby('Housing type')['Stock buildings'].sum()).T.sort_index(ascending=False)
av_ratio_surface


In [ ]:
agg_ratio_surface = generic_input['ratio_surface']
agg_ratio_surface


## Validation: primary energy estimations


#### Taking the aggregate surface ratio used in Res-IRF and no ventilation


In [ ]:
buildings = buildings.set_index('Heating system', append=True)
epc, energy_primary = conventional_energy_3uses(buildings['Wall'], buildings['Floor'], buildings['Roof'], buildings['Windows'], agg_ratio_surface, buildings['Efficiency'], buildings.index)
energy_epc_surface = pd.concat((buildings, epc.rename('calculated_epc'), energy_primary.rename('primary_energy_estimated')), axis=1)


#### Standard Res-IRF estimation: Without taking ventilation into account, but with surface


In [ ]:
epc, energy_primary = conventional_energy_3uses(buildings['Wall'], buildings['Floor'], buildings['Roof'], buildings['Windows'], ratio_surface, buildings['Efficiency'], buildings.index)
energy_epc_std = pd.concat((buildings, epc.rename('calculated_epc')), axis=1)
#energy_epc_std = pd.concat((buildings, epc.rename('calculated_epc'), energy_primary.rename('primary_energy_estimated')), axis=1)
buildings['primary_energy_estimated'] = energy_primary
buildings['estimation_error'] = buildings['primary_energy_estimated'] - buildings['primary energy']


#### Taking ventilation into account


In [ ]:
epc, energy_primary = conventional_energy_3uses(buildings['Wall'], buildings['Floor'], buildings['Roof'], buildings['Windows'], ratio_surface, buildings['Efficiency'], buildings.index, air_rate=buildings['Ventilation'])
energy_epc_vent = pd.concat((buildings, epc.rename('calculated_epc'), energy_primary.rename('primary_energy_estimated_vent')), axis=1)
energy_epc_vent['vent_estimation_error'] = energy_epc_vent['primary_energy_estimated_vent'] - energy_epc_vent['primary energy']


In [ ]:
stock_main = buildings.copy()
stock_main = stock_main.reset_index('Heating system')
stock_other_carriers = stock_main.replace(to_replace={'Electricity-Performance boiler': 'Natural gas-Standard boiler', 'Natural gas-Standard boiler': 'Electricity-Performance boiler', 'Natural gas-Performance boiler': 'Electricity-Performance boiler', 'Wood fuel-Standard boiler': 'Electricity-Performance boiler'}).copy()

# Creating the weight column which gather the share of the selected energy carrier in the heating system, for the main or the others energy carriers
for idx in stock_main.index:
    if stock_main.loc[idx,'Heating system'] in {'Natural gas-Standard boiler',  'Natural gas-Performance boiler', 'Wood fuel-Standard boiler'}:
        stock_main.loc[idx, 'weights 1'] = stock_main.loc[idx, 'Fossil fuel share']
    else:
        stock_main.loc[idx, 'weights 1'] = stock_main.loc[idx, 'Electric fuel share']

for idx in stock_other_carriers.index:
    if stock_other_carriers.loc[idx,'Heating system'] in {'Natural gas-Standard boiler',  'Natural gas-Performance boiler', 'Wood fuel-Standard boiler'}:
        stock_other_carriers.loc[idx, 'weights 2'] = stock_other_carriers.loc[idx, 'Fossil fuel share']
    else:
        stock_other_carriers.loc[idx, 'weights 2'] = stock_other_carriers.loc[idx, 'Electric fuel share']

# Estimation of annual energy needs for the main energy carrier case
stock_main = stock_main.set_index('Heating system', append=True)
_, stock_main['energy_primary_estimate'] = conventional_energy_3uses(stock_main['Wall'], stock_main['Floor'],
                                                                     stock_main['Roof'], stock_main['Windows'],
                                                                     ratio_surface, stock_main['Efficiency'],
                                                                     stock_main.index, air_rate=stock_main['Ventilation'])
# applying the weight
stock_main['energy_primary_weighted'] = stock_main['energy_primary_estimate'] * stock_main['weights 1']
stock_main = stock_main.reset_index('Heating system')

# Estimation of annual energy needs for the case of the other energy carrier
heating_eff = stock_other_carriers['Heating system'].replace(efficiency).set_axis(stock_other_carriers.index)
stock_other_carriers['Efficiency'] = heating_eff.rename('Efficiency')
stock_other_carriers = stock_other_carriers.set_index('Heating system', append=True)
_, stock_other_carriers['energy_primary_estimate'] = conventional_energy_3uses(stock_other_carriers['Wall'], stock_other_carriers['Floor'],
                                                                               stock_other_carriers['Roof'], stock_other_carriers['Windows'],
                                                                               ratio_surface, stock_other_carriers['Efficiency'], stock_other_carriers.index, air_rate=stock_other_carriers['Ventilation'])
# Applying the weight
stock_other_carriers['energy_primary_weighted'] = stock_other_carriers['energy_primary_estimate'] * stock_other_carriers['weights 2']
stock_other_carriers = stock_other_carriers.reset_index('Heating system')


In [ ]:
buildings['primary_energy_weighted'] = stock_main['energy_primary_weighted'] + stock_other_carriers['energy_primary_weighted']
buildings['weighted_estimation_error'] = buildings['primary_energy_weighted'] - buildings['primary energy']


In [ ]:
# check weights
#stock_main['weights 1'] + stock_other_carriers['weights 2']


In [ ]:
mse_weighted = math.sqrt((buildings['weighted_estimation_error']**2).sum()/buildings.shape[0])
mse_standard = math.sqrt((buildings['estimation_error']**2).sum()/buildings.shape[0])
mse_vent = math.sqrt((energy_epc_vent['vent_estimation_error']**2).sum()/energy_epc_vent.shape[0])
print('MSE for the standard estimation: {}'.format(mse_standard))
print('MSE for the weighted estimation: {}'.format(mse_weighted))
print('MSE for estimation which takes ventilation into account: {}'.format(mse_vent))


In [ ]:
colors = {'Single-family': 'red', 'Multi-family': 'blue'}
scatter = buildings.plot.scatter(x = 'primary energy', y = 'primary_energy_weighted', xlim=(50, 700), ylim=(50,700), c=buildings.index.get_level_values('Housing type').map(colors), fontsize=10)
scatter.set_title('Estimation error per housing type (weighted estimation)', fontsize=14)
scatter.set_xlabel('primary energy', fontsize=12)
scatter.set_ylabel('weighted estimation', fontsize=12)
scatter.axis('scaled')
scatter.set(xlim=(50, 700), ylim=(50,700))
scatter = scatter.plot([0, 1], [0, 1], transform=scatter.transAxes, ls='--',  c='green' )


In [ ]:
colors = {'Single-family': 'red', 'Multi-family': 'blue'}
scatter = buildings.plot.scatter(x = 'primary energy', y = 'primary_energy_estimated', c=buildings.index.get_level_values('Housing type').map(colors), fontsize=10)
scatter.set_title('Estimation error per housing type (standard estimation)', fontsize=14)
scatter.set_xlabel('primary energy', fontsize=12)
scatter.set_ylabel('standard estimation', fontsize=12)
scatter.axis('scaled')
scatter.set(xlim=(50, 700), ylim=(50,700))
scatter = scatter.plot([0, 1], [0, 1], transform=scatter.transAxes, ls='--',  c='green' )


In [ ]:
colors = {'Wood fuel-Standard boiler': 'brown', 'Electricity-Performance boiler': 'blue','Natural gas-Standard boiler': 'yellow', 'Natural gas-Performance boiler': 'yellow' }
scatter = buildings.plot.scatter(x = 'primary energy', y = 'primary_energy_weighted', xlim=(50, 700), ylim=(50,700), c=buildings.index.get_level_values('Heating system').map(colors), fontsize=10)
scatter.set_title('Estimation error per main heating energy (weighted estimation)', fontsize=14)
scatter.set_xlabel('primary energy', fontsize=12)
scatter.set_ylabel('weighted estimation', fontsize=12)
scatter.axis('scaled')
scatter.set(xlim=(50, 700), ylim=(50,700))
scatter = scatter.plot([0, 1], [0, 1], transform=scatter.transAxes, ls='--',  c='green' )


In [ ]:
colors = {'Wood fuel-Standard boiler': 'brown', 'Electricity-Performance boiler': 'blue','Natural gas-Standard boiler': 'yellow', 'Natural gas-Performance boiler': 'yellow' }
scatter = buildings.plot.scatter(x = 'primary energy', y = 'primary_energy_estimated', xlim=(50, 700), ylim=(50,700), c=buildings.index.get_level_values('Heating system').map(colors), fontsize=10)
scatter.set_title('Estimation error per housing type (standard estimation)', fontsize=14)
scatter.set_xlabel('primary energy', fontsize=12)
scatter.set_ylabel('standard estimation', fontsize=12)
scatter.axis('scaled')
scatter.set(xlim=(50, 700), ylim=(50,700))
scatter = scatter.plot([0, 1], [0, 1], transform=scatter.transAxes, ls='--',  c='green' )


In [ ]:
surface_check =  buildings.copy()
surface_check['energy_surface_estimation'] = energy_epc_surface['primary_energy_estimated']
colors = {'Wood fuel-Standard boiler': 'brown', 'Electricity-Performance boiler': 'blue','Natural gas-Standard boiler': 'yellow', 'Natural gas-Performance boiler': 'yellow' }
scatter =  surface_check.plot.scatter(x = 'primary energy', y = 'energy_surface_estimation', xlim=(50, 700), ylim=(50,700), c=buildings.index.get_level_values('Heating system').map(colors), fontsize=10)
scatter.set_title('Estimation comparison per heating system (average ratio surface) ', fontsize=14)
scatter.set_xlabel('primary energy', fontsize=12)
scatter.set_ylabel('average surface ratio estimation', fontsize=12)
scatter.axis('scaled')
scatter.set(xlim=(50, 700), ylim=(50,700))
scatter = scatter.plot([0, 1], [0, 1], transform=scatter.transAxes, ls='--',  c='green' )


In [ ]:
estimations = buildings.copy()
estimations['av_surface_estimation'] = energy_epc_surface['primary_energy_estimated']
estimations.loc[:, ('construction period','Ratio_surface_Wall', 'Ratio_surface_Floor', 'Ratio_surface_Roof', 'Ratio_surface_Window' ,'primary energy', 'primary_energy_estimated', 'primary_energy_weighted','av_surface_estimation', 'estimation_error', 'weighted_estimation_error')]


### Discussion
* Wood fueled, Single-family, before 1915 built dwellings are the most over-estimated annual energy need by all methods.
* Comparing the line LC3 to LC4: the change from gas to electricity as a main source of energy for 1915 built multi-family buildings decreases energy need, while our estimations account for a significant increases. U-values doesn't change since the construction period are the same. What changes in our estimation is the efficiency and the surface ratio. When we look at the av_surface_estimation, for which the surface ratio remain the same, the estimation still increases instead of decreasing. I conclude that it's our efficency parameter which gives this error and of course the fact we omit the parameters which causes the original decrease. Our simple weighting doesn't solve the issue.

Observed parameters:
- surface ratio: we calculate using the Res-IRF averages to test
- u values: a large part comes from the French 3CL reference values and the other when reported from profeel
- efficiency: We calculate the weighted average of the annual energy need wrt. electric or fossil fuel share of the heating system. Each fossil fuel has the same efficiency value.


#### For data_matching


In [ ]:
energy_epc_std = energy_epc_std.drop(labels=['Ratio_surface_Wall', 'Ratio_surface_Floor', 'Ratio_surface_Roof','Ratio_surface_Window', 'construction period', 'EPC', 'primary energy', 'Ventilation', 'Ventilation gross', 'Electric fuel share', 'Fossil fuel share', 'Efficiency'], axis=1)


In [ ]:
os.makedirs(os.path.join('data', 'derived', 'profeel'), exist_ok=True)
energy_epc_std.to_csv(os.path.join('data', 'derived', 'profeel', 'building_stock_profeel_std.csv'))
